# Task 1: News Topic Classifier Using BERT

**Objective**  
Fine-tune `bert-base-uncased` to classify news headlines into one of 4 topics:  
World, Sports, Business, Sci/Tech.

**Dataset**  
AG News (amananandrai/ag-news-classification-dataset)  
Train: ~120,000 samples  
Test: ~7,600 samples  
Classes: 4 (balanced)

## Dataset Loading & Preprocessing

- Downloaded via `kagglehub`  
- Combined `Title` + `Description` → `text` column  
- Labels: Class Index (1–4) → 0–3  
- Tokenized with `bert-base-uncased` tokenizer (max_length=128, truncation & padding)

In [1]:
import kagglehub

path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")
print("Dataset downloaded to:", path)

Using Colab cache for faster access to the 'ag-news-classification-dataset' dataset.
Dataset downloaded to: /kaggle/input/ag-news-classification-dataset


In [2]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import os



In [4]:
print("Files in dataset folder:")
print(os.listdir(path))


Files in dataset folder:
['train.csv', 'test.csv']


In [5]:
train_file = os.path.join(path, "train.csv")
test_file = os.path.join(path, "test.csv")

In [6]:
train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

In [7]:

train_df['text'] = train_df['Title'].fillna('') + " " + train_df['Description'].fillna('')
test_df['text'] = test_df['Title'].fillna('') + " " + test_df['Description'].fillna('')

In [8]:
train_df['label'] = train_df['Class Index'] - 1
test_df['label'] = test_df['Class Index'] - 1

print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nSample rows:")
display(train_df.head(3))

print("\nLabel distribution:")
print(train_df['label'].value_counts())


Train shape: (120000, 5)
Test shape: (7600, 5)

Sample rows:


,Class Index,Title,Description,text,label
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...,2
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...,2



Label distribution:
label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64


## Model Development & Training

- Model: `distilbert-base-uncased` + classification head (4 labels)  
- Framework: Hugging Face Trainer API  
- Training settings:  
  - Subset size: 8,000 samples  
  - Epochs: 1  
  - Batch size: 64 (with gradient accumulation)  
  - Trained on Colab GPU  
- Model saved to `./fine_tuned_distilbert_agnews`

In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

print("DistilBERT tokenizer and model loaded (smaller & faster)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT tokenizer and model loaded (smaller & faster)


In [10]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [11]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Re-tokenized with DistilBERT")

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Re-tokenized with DistilBERT


In [12]:
from transformers import TrainingArguments

subset_size = 8000
tokenized_train_small = tokenized_train.shuffle(seed=42).select(range(subset_size))


training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size = 64,
    num_train_epochs = 1,
)

training_args.gradient_accumulation_steps = 2


In [14]:

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=63, training_loss=1.05137937031095, metrics={'train_runtime': 5494.8298, 'train_samples_per_second': 1.456, 'train_steps_per_second': 0.011, 'total_flos': 264944246784000.0, 'train_loss': 1.05137937031095, 'epoch': 1.0})

## Evaluation Metrics

- Accuracy  
- Weighted F1-score (not computed in this run, but can be added)

**Final Test Results**  
(From trainer.evaluate())
- accuracy: [0.8992]  
- eval_loss: [0.3236]  
- eval_runtime: [1604.3338

]

In [15]:
eval_results = trainer.evaluate()

print("\nFinal Evaluation Results on Test Set:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Final Evaluation Results on Test Set:
eval_loss: 0.3236
eval_accuracy: 0.8992
eval_runtime: 1604.3338
eval_samples_per_second: 4.7370
eval_steps_per_second: 0.5920
epoch: 1.0000


In [16]:
trainer.save_model("./fine_tuned_distilbert_agnews")
tokenizer.save_pretrained("./fine_tuned_distilbert_agnews")

print("Model and tokenizer saved to ./fine_tuned_distilbert_agnews")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ./fine_tuned_distilbert_agnews


## Final Summary & Insights

- DistilBERT achieved solid accuracy with minimal training (1 epoch, 8k subset)  
- Fast and lightweight model compared to full BERT  
- Combining title + description helps capture context  
- Limitations: short training & small subset — full run would give higher scores  
- Skills gained: Transformers fine-tuning, Trainer API, Gradio deployment

## Live Demo – Gradio Interface

Interactive tool to classify any news text in real time.  

Enter headline/description → get predicted topic + top 3 confidences.

In [17]:
!pip install gradio -q

In [18]:

import gradio as gr
from transformers import pipeline
import torch


In [19]:
classifier = pipeline(
    "text-classification",
    model="./fine_tuned_distilbert_agnews",
    tokenizer="./fine_tuned_distilbert_agnews",
    device=0 if torch.cuda.is_available() else -1,
    return_all_scores=True
)

id2label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

def predict(text):
    if not text.strip():
        return "Please enter some news text!"
    results = classifier(text)[0]
    sorted_results = sorted(results, key=lambda x: x['score'], reverse=True)

    top1 = sorted_results[0]
    top3 = sorted_results[:3]

    output = f"**Predicted Topic:** {id2label[int(top1['label'].split('_')[-1])]} (Confidence: {top1['score']:.4f})\n\n"
    output += "**Top 3 Predictions:**\n"
    for i, res in enumerate(top3, 1):
        label_idx = int(res['label'].split('_')[-1])
        output += f"{i}. {id2label[label_idx]} ({res['score']:.4f})\n"

    return output

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [20]:

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(
        label="Enter news headline or description",
        placeholder="Example: Apple launches new AI-powered MacBook...",
        lines=5
    ),
    outputs=gr.Textbox(label="Prediction Results"),
    title="News Topic Classifier (DistilBERT on AG News)",
    description="Type any news text to see the predicted category.",
    examples=[
        "Pakistan cricket team wins T20 World Cup final",
        "Stock market crashes amid global recession fears",
        "Iran has dropped missiles on Isreal beack to back ",
        "Tesla announces breakthrough in solid-state battery technology"
    ],
    theme="soft"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://be2ccadd29d7160750.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Final Summary & Insights

- DistilBERT achieved solid accuracy with minimal training (1 epoch, 8k subset)  
- Fast and lightweight model compared to full BERT  
- Combining title + description helps capture context  
- Limitations: short training & small subset — full run would give higher scores  
- Skills gained: Transformers fine-tuning, Trainer API, Gradio deployment